# Лекция 09. Области видимости и декораторы

Функция может получить другую функцию, сохранить её и вернуть новую. На этом строятся настраиваемые проверки, независимые счётчики и обёртки, которые добавляют журналирование к готовому коду. Разберём, откуда такая обёртка берёт имена и что именно делает запись с `@`.

## Цели

После лекции вы сможете:

- передавать функции как значения и отличать функцию от результата её вызова;
- находить имя в локальной, внешней и модульной областях видимости;
- объяснять `UnboundLocalError`, назначение `global` и `nonlocal`;
- создавать замыкания с настройками и независимым состоянием;
- писать обёртки, сохраняющие аргументы, результат и исключения функции;
- выбирать декораторы для логов, повторных попыток, реестров и настройки поведения;
- сохранять допустимые способы вызова исходной функции и понимать границы `functools.wraps`;
- различать создание декоратора, декорирование и вызов готовой функции.

## Перед началом

Нужны функции, аргументы и `return`, контейнеры, ссылки на объекты, импорт и исключения из занятий 1–8. Классы, генераторы и асинхронность не используются. Все примеры работают на Python 3.14 со стандартной библиотекой, без файлов и интернета.

Запускайте код сверху вниз. Намеренно ошибочные варианты либо перехвачены в демонстрации, либо отдельно показаны в тексте. После изменения примеров перезапустите ядро: старые имена в ноутбуке могут скрыть ошибку.

## Функция — объект, а имя — ссылка на него

В `sorted(..., key=...)` мы уже передавали функцию другому коду. Пока после имени нет скобок вызова, мы работаем с самим объектом функции.

Присваивание нового имени не запускает функцию и не копирует её код. Оба имени могут ссылаться на одну функцию; вызвать её можно через любое из них.

In [ ]:
def normalize_name(text: str) -> str:
    return text.strip().lower()


normalize_alias = normalize_name
assert normalize_alias is normalize_name
assert normalize_alias("  ЧАЙ  ") == "чай"
print(normalize_name)
print(normalize_name("  ЧАЙ  "))

## Передать правило вместо готового результата

Напишем обработку, которой можно передать способ преобразования строки. Параметр `transform` — локальное имя полученной функции. Внутри цикла мы вызываем её, когда появились нужные аргументы.

`transform=normalize_name` передаёт правило. `transform=normalize_name("чай")` заранее вычислит строку и передаст её вместо функции: последующий вызов `transform(value)` уже не сработает.

В заданиях встретится аннотация `Callable` из `collections.abc`: `Callable[[int], bool]` означает функцию, которая принимает одно целое число и возвращает `bool`. `Callable` без уточнений обозначает вызываемый объект без перечисления его параметров.

In [ ]:
def transform_names(values: list[str], transform) -> list[str]:
    result = []
    for value in values:
        result.append(transform(value))
    return result


assert transform_names(["  ЧАЙ ", " ХЛЕБ"], normalize_name) == ["чай", "хлеб"]
assert transform_names(["чай", "хлеб"], str.upper) == ["ЧАЙ", "ХЛЕБ"]

## Одинаковые имена могут обозначать разные объекты

При вызове функции её параметры и локальные переменные получают собственные привязки. Локальное присваивание не меняет одноимённую переменную модуля.

В ноутбуке верхний уровень ячеек играет роль модульной области: определения из предыдущих выполненных ячеек доступны следующим. У каждого импортированного модуля своя такая область. `global` не означает одно общее пространство всех модулей программы.

In [ ]:
caption = "Каталог"


def make_caption(name: str) -> str:
    caption = "Товар"
    return f"{caption}: {name}"


assert make_caption("чай") == "Товар: чай"
assert caption == "Каталог"

## Где Python ищет имя: LEGB

Для обычного чтения имени внутри наших функций удобно правило **LEGB**:

1. **Local** — параметры и локальные имена текущего вызова.
2. **Enclosing** — области внешних функций, в которых функция определена.
3. **Global** — область модуля, где функция определена.
4. **Builtins** — встроенные имена: `len`, `int`, `print` и другие.

Поиск заканчивается на первой подходящей привязке. Если имени нигде нет, возникает `NameError`. Здесь важна вложенность **определений**, а не цепочка функций, из которой нас вызвали.

![Имя ищется в текущей функции, внешних функциях, модуле и встроенных именах.](assets/name-resolution.svg)

`if`, `for` и `with` сами по себе не создают новую область видимости имён. Например, счётчик `for` внутри функции остаётся её локальным именем. Области классов и специальные случаи языка изучаются отдельно.

## Почему чтение может дать `UnboundLocalError`

Если в теле функции есть присваивание имени и нет объявления `global` или `nonlocal`, Python считает это имя локальным во всей функции. Чтение до первого присваивания не переключается на одноимённую переменную модуля.

В `counter += 1` сначала нужно прочитать старое значение, а потом привязать новое. У локального `counter` в примере ещё нет значения.

In [ ]:
counter = 10


def broken_increment():
    counter += 1
    return counter


try:
    broken_increment()
except UnboundLocalError:
    print("Локальный counter читается до присваивания")
assert counter == 10

## `global`: явно менять привязку в модуле

Объявление `global counter` говорит: присваивания этому имени в функции относятся к модулю. Для простого чтения модульного имени `global` не требуется.

Так можно вести общий счётчик, но все пользователи функции будут делить одно состояние. Если каждой обработке нужен собственный счётчик, стоит создать отдельное замыкание.

In [ ]:
def increment_global() -> int:
    global counter
    counter += 1
    return counter


assert increment_global() == 11
assert increment_global() == 12
assert counter == 12

## Вернуть функцию вместе с нужными ей привязками

Внутри функции можно определить другую функцию и вернуть её. Внутренняя функция продолжает пользоваться именами внешней функции даже после того, как внешний вызов закончился. Это **замыкание**.

Ниже каждый вызов `make_labeler` создаёт своё имя `prefix` и возвращает новую функцию `label`. У `label` есть собственный параметр `name`; имя `prefix` находится во внешней функции, `separator` — в модуле, а `len` — среди встроенных имён.

In [ ]:
separator = ": "


def make_labeler(prefix: str):
    def label(name: str) -> str:
        clean_name = name.strip()
        return f"{prefix}{separator}{clean_name} ({len(clean_name)})"
    return label


storage_label = make_labeler("Склад")
shop_label = make_labeler("Магазин")
assert storage_label is not shop_label
assert storage_label(" чай ") == "Склад: чай (3)"
assert shop_label("чай") == "Магазин: чай (3)"

## Замыкание может хранить изменяющееся состояние

Для чтения `prefix` ничего объявлять не потребовалось. Для изменения привязки из внешней функции используют `nonlocal`.

`nonlocal` выбирает уже существующее имя в ближайшей подходящей внешней функции. Оно не создаёт переменную модуля. Если во внешних функциях такого имени нет, определение с `nonlocal` некорректно.

![Два вызова фабрики создают два независимых счётчика; повторные вызовы одного замыкания меняют только его состояние.](assets/closure-state.svg)

In [ ]:
def make_counter(start: int = 0):
    current = start

    def next_value() -> int:
        nonlocal current
        current += 1
        return current

    return next_value


first_counter = make_counter()
second_counter = make_counter(100)
assert first_counter() == 1
assert first_counter() == 2
assert second_counter() == 101
assert first_counter() == 3

Снова различим **перепривязку имени** и **изменение объекта** из лекции 5. `current += 1` для целого числа создаёт новую привязку, поэтому нужен `nonlocal`. Если замыкание делает `events.append("called")`, оно меняет список, а имя `events` не перепривязывает: `nonlocal` для такого действия не нужен.

Замыкание сохраняет доступ к привязке, а не обещает заморозить значение навсегда. Изменения доступного ему общего объекта или модульного имени могут повлиять на следующий вызов. Независимость счётчиков получилась потому, что каждый вызов фабрики создал собственное состояние.

## Зачем нужны декораторы

Есть полезная функция: она разбирает строку, строит подпись или читает данные. Вокруг неё постепенно появляется служебная работа: записать начало и исход вызова, повторить действие после временного сбоя, добавить функцию в список обработчиков. Если переписать эту работу в каждой функции, исправлять её тоже придётся во многих местах.

Декоратор позволяет один раз описать такое действие и применять его к выбранным функциям рядом с их определениями. Основной алгоритм и добавленное поведение можно читать и проверять отдельно.

| Задача | Что делает декоратор | Что остаётся в исходной функции |
| --- | --- | --- |
| Логи | Записывает имя операции, начало, завершение и ошибку | Сама обработка данных |
| Повторные попытки | После оговорённого временного сбоя повторяет вызов в пределах лимита | Одна попытка выполнить операцию |
| Реестр обработчиков | При определении помещает функцию в словарь под выбранным именем | Обработка одного формата или команды |
| Настройка поведения | По параметрам выбирает политику: число попыток, точность результата, метку журнала | Вычисление с обычными аргументами |

В первых примерах обёртка добавит журнал. Затем тот же приём применим к повторным вызовам и настройкам. У реестра другой вариант: декоратор может выполнить регистрацию и вернуть саму исходную функцию, вообще не создавая обёртку.

Параметры декоратора задают общую политику, а аргументы декорированной функции — данные конкретного вызова. Например, число попыток задают при настройке декоратора; путь к читаемому файлу по-прежнему передают функции чтения.

## Добавить действие к готовой функции

Допустим, перед разбором количества нужно записывать начало работы, а при любом выходе — её окончание. Такие действия могут понадобиться многим функциям. Обёртка берёт на себя общую часть, вызывает исходную функцию и передаёт её результат дальше.

Ей нужно сохранить ссылку на исходную функцию. Это уже знакомая задача для замыкания: внешняя функция получает `func`, внутренняя пользуется этим именем при последующих вызовах.

Чтобы одна обёртка подошла функциям с разными параметрами, сначала разберём передачу аргументов.

## `*args` и `**kwargs`: принять и передать аргументы

В **определении** функции `*args` собирает дополнительные позиционные аргументы в кортеж, `**kwargs` — именованные аргументы в словарь. Имена `args` и `kwargs` — соглашение; сбор выполняют звёздочки.

В **вызове** те же записи распаковывают кортеж и словарь. Поэтому `func(*args, **kwargs)` передаёт полученные аргументы исходной функции. Звёздочки здесь работают в обратную сторону.

Отдельная `*` в сигнатуре, например `def label(name, *, suffix="")`, делает следующие параметры только именованными. Такую функцию вызывают как `label("чай", suffix="!")`; обёртка должна передать `suffix` через `**kwargs`.

In [ ]:
def describe_arguments(*args, **kwargs):
    return args, kwargs


assert describe_arguments(2, 3, unit="кг") == ((2, 3), {"unit": "кг"})

values = ("чай", "хлеб")
options = {"sep": "; "}
print(*values, **options)

## Собираем обёртку без специального синтаксиса

`trace` принимает функцию и возвращает другую функцию. Вызов `trace(parse_quantity)` ещё не разбирает количество: он только создаёт обёртку.

Сама `wrapper` вызывается позже. Она начинает запись, возвращает результат исходной функции через `return` и выполняет завершающую запись через `finally`. Исключение из исходной функции продолжает распространяться — обёртка его не подавляет.

In [ ]:
def trace(func):
    def wrapper(*args, **kwargs):
        print("Начало вызова")
        try:
            return func(*args, **kwargs)
        finally:
            print("Конец вызова")
    return wrapper


def parse_quantity(text: str, base: int = 10) -> int:
    return int(text, base)


original_parser = parse_quantity
traced_parser = trace(original_parser)
assert traced_parser is not original_parser
assert traced_parser("12") == 12
assert traced_parser("10", base=16) == 16
try:
    traced_parser("много")
except ValueError:
    print("ValueError дошёл до вызывающего кода")

## Декоратор и запись с `@`

Декоратор получает функцию и возвращает объект, который будет связан с её именем. В примерах с журналом этот объект — функция-обёртка; декоратор регистрации ниже вернёт саму исходную функцию.

```python
@trace
def parse_quantity(text, base=10):
    return int(text, base)
```

Для этого примера смысл тот же, что у обычного определения и следующего присваивания:

```python
def parse_quantity(text, base=10):
    return int(text, base)

parse_quantity = trace(parse_quantity)
```

Вариант с `@` связывает декорирование с определением функции. Декоратор применяется, когда выполняется определение, а не при каждом последующем вызове. Ранее сохранённая ссылка на исходную функцию сама по себе не меняется.

![Декоратор получает исходную функцию и возвращает обёртку; при вызове обёртка обращается к сохранённой исходной функции.](assets/decorator-flow.svg)

In [ ]:
@trace
def format_item(name: str, quantity: int = 1) -> str:
    return f"{name};{quantity}"


assert format_item("чай", quantity=2) == "чай;2"
assert original_parser("7") == 7  # Эта ссылка по-прежнему ведёт к исходной функции.

## `functools.wraps`: сохранить сведения о функции

После декорирования имя `format_item` ведёт к обёртке. У обычной обёртки собственное имя — `wrapper`, собственная документация. Это неудобно для справки и диагностики.

`@wraps(func)` переносит основные сведения об исходной функции на обёртку, включая имя и строку документации. Оно не выполняет за нас вызов `func` и не добавляет забытый `return`: поведение обёртки по-прежнему определяет её код.

In [ ]:
from functools import wraps


def traced(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Начало: {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"Конец: {func.__name__}")
    return wrapper


@traced
def read_quantity(text: str, base: int = 10) -> int:
    """Преобразовать текст в количество."""
    return int(text, base)


assert format_item.__name__ == "wrapper"
assert read_quantity.__name__ == "read_quantity"
assert read_quantity.__doc__ == "Преобразовать текст в количество."
assert read_quantity("10", base=16) == 16

## Служебный декоратор не должен ломать сигнатуру

Сигнатура описывает параметры функции: их имена, порядок, способы передачи и значения по умолчанию. Когда к функции добавляют журналирование или повторные попытки, вызывающий код должен сохранить возможность вызывать её прежним способом.

Если до декорирования работали `read_quantity("10")`, `read_quantity("10", base=16)` и `read_quantity(text="10", base=16)`, все эти формы должны работать и после. Добавление обязательного параметра обёртки, переименование параметра или потеря поддержки именованных аргументов ломает публичный интерфейс. Для служебного декоратора это плохое проектное решение.

Настройки самого декоратора передают его фабрике: `@retry(attempts=3)`. Требовать из-за этого новый аргумент в каждом вызове исходной функции не следует.

У универсальной обёртки запись `wrapper(*args, **kwargs)` отличается от сигнатуры исходной функции. Это нормально: обёртка принимает и пересылает аргументы. Сохранить нужно **допустимые способы вызова**, а не буквально повторить текст определения.

Покажем намеренно плохую обёртку: она оставила только параметр `text`. Даже `@wraps` её поведение не исправит.

In [ ]:
from inspect import signature


def narrowed(func):
    @wraps(func)
    def wrapper(text):
        return func(text)
    return wrapper


broken_reader = narrowed(original_parser)
assert broken_reader("10") == 10
try:
    broken_reader("10", base=16)
except TypeError:
    print("Допустимый раньше вызов сломан: wrapper не принимает base")

print("Сигнатура для справки:", signature(broken_reader))
print("Сама обёртка:", signature(broken_reader, follow_wrapped=False))

good_reader = traced(original_parser)
assert good_reader("10", base=16) == 16
assert good_reader(text="10", base=16) == 16
assert signature(good_reader) == signature(original_parser)

`wraps` сохраняет ссылку на исходную функцию в `__wrapped__`. По умолчанию `inspect.signature()` следует по этой ссылке и показывает её сигнатуру. С `follow_wrapped=False` можно увидеть параметры самой обёртки.

Поэтому одной красивой сигнатуры в справке недостаточно: плохая `wrapper(text)` всё равно отвергла `base`. Корректность обеспечили `*args`, `**kwargs` и их передача в `func`, а `wraps` сохранил сведения для инструментов и человека.

Явный адаптер может специально вводить другой интерфейс; тогда это отдельный, документированный договор. Добавление логов или повторов само по себе не должно заставлять переписывать вызывающий код.

## Декоратор с настройкой — ещё одна функция снаружи

Теперь журналу нужна метка, общая для нескольких вызовов. Запись `@announced("Импорт")` сначала вызывает фабрику с меткой. Фабрика возвращает декоратор; тот получает исходную функцию и возвращает обёртку.

![Три этапа: фабрика получает настройку, декоратор получает функцию, обёртка получает аргументы вызова.](assets/decorator-factory.svg)

На каждом уровне свои аргументы:

- `announced(label)` — настройка, когда выполняется выражение декоратора;
- `decorate(func)` — исходная функция, когда происходит декорирование;
- `wrapper(*args, **kwargs)` — данные очередного вызова.

Метка и исходная функция доступны обёртке через замыкание. При каждом вызове не нужно заново создавать ни фабрику, ни декоратор.

In [ ]:
def announced(label: str):
    def decorate(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"{label}: {func.__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorate


@announced("Импорт")
def parse_amount(text: str, base: int = 10) -> int:
    return int(text, base)


assert parse_amount("12") == 12
assert parse_amount("10", base=16) == 16
assert parse_amount.__name__ == "parse_amount"

## Применения: журнал, повторы, реестр и настройки

### Логи: что случилось с вызовом

Наш `traced` уже показывает простейший журнал: начало и окончание вызова. Можно также записывать успешное завершение и тип ошибки; после записи ошибки повторный `raise` передаст её дальше. В приложении такие записи обычно отправляют через модуль `logging`; в учебном примере достаточно `print`.

Служебный журнал особенно полезен, когда ошибка приходит из нескольких уровней вызовов. Он добавляет сведения о ходе работы, сохраняя результат и ошибку исходной операции.

### Повтор: несколько попыток выполнить одну операцию

После временного сбоя иногда можно повторить операцию. Декоратор хранит общую политику: сколько всего попыток разрешено и на какую ошибку реагировать. Функция описывает одну попытку.

Ниже повторяем только после `TimeoutError`, не больше `attempts` раз **всего**, включая первый вызов. Другие исключения сразу выходят наружу. На последней неудаче повторный `raise` сохраняет исходную ошибку. Счётчик попыток начинается заново для каждого вызова обёртки.

В определении `def retry(*, attempts: int):` одиночная `*` означает, что настройка передаётся только по имени. Так же устроен параметр `digits` у фабрики `rounded_result` ниже.

In [ ]:
def retry(*, attempts: int):
    if attempts < 1:
        raise ValueError("Нужна хотя бы одна попытка")

    def decorate(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except TimeoutError:
                    if attempt == attempts:
                        raise
        return wrapper
    return decorate


attempt_log = []


@retry(attempts=3)
def read_fragment(name: str, suffix: str = "") -> str:
    attempt_log.append(name)
    if len(attempt_log) < 3:
        raise TimeoutError("Временный сбой чтения")
    return name + suffix


assert read_fragment("отчёт", suffix="!") == "отчёт!"
assert attempt_log == ["отчёт", "отчёт", "отчёт"]

Это локальная имитация: сеть и ожидания для запуска не нужны. Повторяющиеся вызовы видны по списку `attempt_log`.

Повтор требует подходящей операции. Если сервис уже выполнил платёж, а ответ потерялся, повторный вызов может провести второй платёж. Поэтому такую политику применяют там, где повтор безопасен, либо операция имеет защиту от повторного выполнения. Произвольную ошибку программы повтором не исправить.

### Реестр: найти функцию по имени

Программа может выбирать обработчик по строке из настроек или входных данных. Сохраним обработчики в словаре. Декоратор выполнит регистрацию при определении, проверит, что имя свободно, и вернёт **ту же функцию**.

In [ ]:
formatters = {}


def register_format(name: str):
    def decorate(func):
        if name in formatters:
            raise ValueError(f"Формат {name!r} уже зарегистрирован")
        formatters[name] = func
        return func
    return decorate


@register_format("plain")
def plain_name(name: str) -> str:
    return name


@register_format("upper")
def upper_name(name: str) -> str:
    return name.upper()


assert formatters["plain"] is plain_name
assert formatters["upper"] is upper_name
selected_format = "upper"
assert formatters[selected_format]("чай") == "ЧАЙ"

При регистрации `plain_name` и `upper_name` не вызывались. Словарь получил ссылки на них, а выбранный обработчик запустился позже. Здесь декоратор нужен для объявления участия в реестре; `wraps` не требуется, поскольку новой обёртки нет.

### Параметры: одна механика, разные правила

`attempts=3` задаёт число попыток, имя формата — место в реестре, а метка `announced` — текст журнала. Настройка может менять и обработку результата. Например, фабрика ниже задаёт точность округления, сохраняя способ вызова исходного вычисления.

In [ ]:
def rounded_result(*, digits: int):
    if digits < 0:
        raise ValueError("Число знаков должно быть неотрицательным")

    def decorate(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return round(func(*args, **kwargs), digits)
        return wrapper
    return decorate


@rounded_result(digits=2)
def per_item(total: float, count: int = 1) -> float:
    return total / count


assert per_item(10, count=3) == 3.33
assert per_item(total=10, count=4) == 2.5

У `per_item` остались параметры `total` и `count` с прежними способами передачи. Число знаков хранится в замыкании декоратора. Теперь договор результата явно включает округление; добавлять `digits` в каждый вызов вычисления не требуется.

Сначала стоит назвать добавляемую политику, затем решить, к каким функциям её применять и какие настройки нужны. Декоратор удобен, когда это самостоятельное повторяющееся правило.

## Что обёртка должна сохранить

При добавлении журнала договор исходной функции остаётся прежним:

- все прежние допустимые формы вызова сохраняются: позиционные и именованные аргументы, значения по умолчанию и параметры только по имени;
- результат возвращается вызывающему коду, включая `0`, `False` и `None`;
- исключение доходит до вызывающего, если декоратор специально не вводит другую политику;
- обязательные завершающие действия выполняются и при ошибке;
- каждая обёртка хранит именно свою исходную функцию и настройки.

Забытый `return func(...)` молча заменит результат на `None`. `except Exception: return None` изменит ошибку на обычный возврат. Обе ошибки особенно легко пропустить, если проверять декоратор только на функции, которая печатает текст.

Настройки служебного поведения относятся к фабрике декоратора. Менять ради них публичную сигнатуру исходной функции не следует; `wraps` не исправляет ошибки передачи аргументов.

Для журналирования полезны `finally` и повторный `raise` из лекции 8. Если декоратор переводит сбой на другой уровень задачи, причину можно сохранить через `raise ... from ...`.

## Неожиданно, но по правилам

### 1. Чужой вызов не создаёт внешнюю область для нашей функции

Модульное имя ищется там, где функция определена. Локальные переменные того, кто её вызвал, в эту цепочку не входят.

In [ ]:
place = "модуль"


def read_place() -> str:
    return place


def call_with_local_place() -> str:
    place = "вызывающая функция"
    return read_place()


assert call_with_local_place() == "модуль"

### 2. Декорирование уже произошло до первого вызова

Функция-декоратор может выполнить действие сразу. Действия внутри возвращённой обёртки будут выполняться при последующих вызовах. Перед запуском разделим записи `decorated` и `called` по этим двум моментам.

In [ ]:
decoration_events = []


def mark_definition(func):
    decoration_events.append("decorated")

    @wraps(func)
    def wrapper(*args, **kwargs):
        decoration_events.append("called")
        return func(*args, **kwargs)
    return wrapper


@mark_definition
def answer() -> int:
    return 42


assert decoration_events == ["decorated"]
assert answer() == 42
assert answer() == 42
assert decoration_events == ["decorated", "called", "called"]

## Итоги

Декоратор отделяет повторяющуюся политику от основного алгоритма: журналирует вызов, ограниченно повторяет попытку, регистрирует обработчик или применяет выбранную настройку. Служебная обёртка должна сохранять допустимые способы вызова исходной функции.

Мы начали с обычного объекта функции. Из правила поиска имён получили замыкание, а из замыкания, хранящего другую функцию, — обёртку. Декоратор применяет такое преобразование к определению функции; запись `@` делает это действие явным рядом с определением.

| Вопрос | На что смотреть |
| --- | --- |
| Почему читается именно это значение? | Где имя локально и в каких внешних определениях его ищут |
| Куда идёт присваивание? | Локальное имя, `global` или `nonlocal` |
| Что хранит замыкание? | Доступ к нужным привязкам внешней функции |
| Что возвращает фабрика? | Функцию с настройками или собственным состоянием |
| Что возвращает декоратор? | Объект, который займёт имя исходной функции |
| Сохранился ли договор вызова? | Прежние формы вызова, значения по умолчанию, результат, политика исключений и сведения о функции |

Обсудим: декоратор пишет «вызов окончен» только после `return func(...)`. Что изменится при исключении? А если он вызывает `func(...)`, но забывает вернуть результат?

Дальше — [семинар](seminar.ipynb) и [домашние задачи](tasks.md). [Стек и очередь](../reference/stack-queue.ipynb) вынесены в отдельный практикум для семинаров.

## Дополнительно: порядок обёрток и позднее связывание

Следующие два примера можно разобрать после основной части или самостоятельно. Они используют те же правила; для обязательных задач не нужен отдельный каталог приёмов с декораторами.

### Несколько декораторов

Для `@outer` над `@inner` итог соответствует `function = outer(inner(function))`: при декорировании ближайший к `def` применяется первым. При обычном вызове сначала начинает работу внешняя обёртка. Вход и выход расположены в обратном порядке.

In [ ]:
def tagged(label: str):
    def decorate(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"Вход {label}")
            try:
                return func(*args, **kwargs)
            finally:
                print(f"Выход {label}")
        return wrapper
    return decorate


@tagged("внешняя")
@tagged("внутренняя")
def double(value: int) -> int:
    return value * 2


assert double(3) == 6

### Замыкание читает привязку, когда его вызывают

В одной фабрике создадим несколько функций в цикле. Имя `prefix` у них общее: цикл не создаёт новую область для каждой итерации. К моменту вызова всех функций оно содержит последнее значение.

Во втором варианте каждый вызов `make_greeting` создаёт собственную привязку. Это тот же способ разделить настройки, который мы уже применили к счётчикам.

In [ ]:
def make_shared_greetings():
    greetings = []
    for prefix in ["Привет", "Добрый день"]:
        def greeting(name: str) -> str:
            return f"{prefix}, {name}"
        greetings.append(greeting)
    return greetings


shared_greetings = make_shared_greetings()
assert shared_greetings[0]("Маша") == "Добрый день, Маша"
assert shared_greetings[1]("Маша") == "Добрый день, Маша"


def make_greeting(prefix: str):
    def greeting(name: str) -> str:
        return f"{prefix}, {name}"
    return greeting


separate_greetings = [make_greeting("Привет"), make_greeting("Добрый день")]
assert separate_greetings[0]("Маша") == "Привет, Маша"
assert separate_greetings[1]("Маша") == "Добрый день, Маша"

## Источники

- [Модель выполнения Python 3.14](https://docs.python.org/3.14/reference/executionmodel.html#resolution-of-names): привязки имён и области видимости.
- [Определения функций — Python 3.14](https://docs.python.org/3.14/reference/compound_stmts.html#function-definitions): аргументы и применение декораторов.
- [`global` и `nonlocal` — Python 3.14](https://docs.python.org/3.14/reference/simple_stmts.html#the-nonlocal-statement): изменение привязок вне текущей функции.
- [`functools.wraps` — Python 3.14](https://docs.python.org/3.14/library/functools.html#functools.wraps): сведения об исходной функции на обёртке.
- [`inspect.signature` — Python 3.14](https://docs.python.org/3.14/library/inspect.html#inspect.signature): сигнатура исходной функции и обёртки, `follow_wrapped`.
- [Модуль `logging` — Python 3.14](https://docs.python.org/3.14/library/logging.html): журнал событий и ошибок приложения.
- [Повтор запросов — HTTP Semantics, RFC 9110](https://www.rfc-editor.org/rfc/rfc9110.html#section-9.2.2): условия безопасного повторения операций.
- [Замыкания внутри цикла — Python FAQ](https://docs.python.org/3.14/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result): отложенное чтение общей привязки.

## Пример экзаменационного вопроса

**Тип:** что выведет код и почему.

**Вопрос.** Декоратор добавляет к результату функции номер её вызова. Что напечатает программа? Почему второй вызов продолжает счётчик первого, хотя сделан через другое имя?

```python
def numbered(func):
    count = 0

    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        return count, func(*args, **kwargs)

    return wrapper


@numbered
def twice(value):
    return value * 2


alias = twice
print(twice(3))
print(alias(value=4))
```

**Пример полного ответа.**

```text
(1, 6)
(2, 8)
```

При выполнении определения с `@numbered` исходная функция `twice` передаётся в `numbered`. Там один раз создаётся `count = 0` и возвращается `wrapper`. Имя `twice` теперь связано с этой обёрткой: смысл тот же, что у присваивания `twice = numbered(twice)` после обычного определения.

Обёртка сохраняет доступ к `func` и `count` из внешней функции. Объявление `nonlocal count` направляет присваивание к этой внешней привязке. При первом вызове счётчик становится равен `1`, исходная функция получает `3` и возвращает `6`. Обёртка возвращает пару `(1, 6)`.

Присваивание `alias = twice` не создаёт новую обёртку: оба имени ссылаются на одну функцию с одним состоянием. Поэтому при вызове через `alias` тот же счётчик становится равен `2`. Аргумент `value=4` проходит через `**kwargs` к исходной функции, которая возвращает `8`. Получается пара `(2, 8)`. При первом вызове позиционный аргумент передавался через `*args`.

`count = 0` заново не выполняется: эта строка находится в `numbered`, а последующие вызовы через `twice` и `alias` запускают только уже созданную `wrapper`.

**Пример неполного ответа.**

«Программа выведет `(1, 6)` и `(2, 8)`: было два вызова, а числа `3` и `4` удваиваются».

**Чего не хватает.** Вывод указан верно, но не объяснено, где хранится счётчик, почему он не обнуляется при каждом вызове и почему другое имя не создаёт независимое состояние. Для полного ответа нужно связать применение декоратора, замыкание, `nonlocal` и две ссылки на одну обёртку.
